<a href="https://colab.research.google.com/github/MPMauricio/Calendarizacion-Ujieres-Antigravity2026/blob/main/pdvconsultahistoricaabril.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================
# 📋 PDV CONSULTA HISTÓRICA - DATOS INCRUSTADOS
# ============================================
!pip install pandas openpyxl -q

import pandas as pd
from datetime import datetime, timedelta
import json
import os
import zipfile

print("="*70)
print(" PDV CONSULTA HISTÓRICA - PROCESANDO BASE DE DATOS")
print("="*70)

from google.colab import files
print("\n📤 Selecciona tu archivo Excel con los PDVs:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Leer Excel
df_temp = pd.read_excel(filename)
df_temp = df_temp.dropna(how='all')

print(f"\n📊 Total de registros cargados: {len(df_temp)}")

# Mapeo de columnas
mapeo_cols = {
    'Ruta': 'Ruta',
    'Nombre del PDV': 'Nombre del PDV',
    'Codigo de PDV': 'Codigo de PDV',
    'Numero de Recarga': 'Numero de Recarga',
    'Tipo Producto': 'Tipo Producto',
    'Categoria': 'Categoria',
    'Fecha Creación': 'Fecha Creación',
    'Distrito': 'Distrito',
    'Departamento': 'Departamento',
    'Municipio': 'Municipio',
    'TIPO DE FACTURA': 'TIPO DE FACTURA',
    'Tipo Negocio': 'Tipo Negocio'
}

cols_existentes = df_temp.columns.tolist()
df_limpio = pd.DataFrame()

for nuevo, original in mapeo_cols.items():
    col_encontrada = None
    for col in cols_existentes:
        if col.strip().lower() == original.strip().lower():
            col_encontrada = col
            break
    if col_encontrada:
        df_limpio[nuevo] = df_temp[col_encontrada].fillna('N/A')
    else:
        df_limpio[nuevo] = 'N/A'

# Limpiar textos
for col in df_limpio.columns:
    df_limpio[col] = df_limpio[col].apply(lambda x: str(x).strip() if pd.notna(x) else 'N/A')

total_pdvs = len(df_limpio)
print(f"✅ {total_pdvs} PDVs procesados correctamente")

# Convertir a JSON para incrustar en HTML
pdv_data = []
for _, row in df_limpio.iterrows():
    pdv_data.append({
        'ruta': str(row['Ruta']),
        'nombre': str(row['Nombre del PDV']),
        'codigo': str(row['Codigo de PDV']),
        'recarga': str(row['Numero de Recarga']),
        'tipo_producto': str(row['Tipo Producto']),
        'categoria': str(row['Categoria']),
        'fecha_creacion': str(row['Fecha Creación']),
        'distrito': str(row['Distrito']),
        'departamento': str(row['Departamento']),
        'municipio': str(row['Municipio']),
        'tipo_factura': str(row['TIPO DE FACTURA']),
        'tipo_negocio': str(row['Tipo Negocio'])
    })

# Convertir a JSON string
pdv_json = json.dumps(pdv_data, ensure_ascii=False)

print("\n📦 Preparando archivo HTML con datos incrustados...")

# ────────────────────────────────────────────
# CREAR HTML CON DATOS INCRUSTADOS
# ────────────────────────────────────────────
html_content = f'''<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no">
    <title>PDV Consulta Histórica</title>
    <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&display=swap" rel="stylesheet">
    <style>
        :root {{ --primary: #4361ee; --secondary: #3f37c9; --success: #06d6a0; --warning: #ffd166; --bg: #f8f9fa; --card: #ffffff; --text: #2b2d42; --gray: #6c757d; }}
        * {{ box-sizing: border-box; margin: 0; padding: 0; }}
        body {{ font-family: 'Poppins', sans-serif; background: var(--bg); color: var(--text); padding-bottom: 20px; }}

        /* LOGIN DISCRETO */
        #loginOverlay {{ position: fixed; top: 0; left: 0; right: 0; bottom: 0; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); z-index: 9999; display: flex; align-items: center; justify-content: center; padding: 20px; }}
        .login-box {{ background: white; border-radius: 16px; padding: 25px; width: 100%; max-width: 320px; box-shadow: 0 15px 50px rgba(0,0,0,0.25); text-align: center; }}
        .login-icon {{ font-size: 40px; margin-bottom: 12px; }}
        .login-title {{ font-size: 18px; font-weight: 700; margin-bottom: 3px; color: var(--text); }}
        .login-subtitle {{ color: var(--gray); font-size: 12px; margin-bottom: 20px; }}
        .login-input {{ width: 100%; padding: 10px 12px; margin-bottom: 8px; border: 2px solid #e0e0e0; border-radius: 8px; font-size: 14px; font-family: 'Poppins', sans-serif; text-align: center; }}
        .login-input:focus {{ outline: none; border-color: var(--primary); }}
        .login-btn {{ width: 100%; padding: 10px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border: none; border-radius: 8px; font-size: 15px; font-weight: 600; cursor: pointer; margin-top: 8px; }}
        .login-error {{ background: #ffebee; color: #c62828; padding: 6px 10px; border-radius: 6px; margin-bottom: 10px; font-size: 11px; display: none; }}
        .login-footer {{ margin-top: 15px; font-size: 10px; color: var(--gray); }}

        /* HEADER */
        #appContent {{ display: none; }}
        .header {{ background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 18px 15px; text-align: center; color: white; border-radius: 0 0 20px 20px; margin-bottom: 15px; position: relative; }}
        .header h1 {{ font-size: 17px; font-weight: 700; margin-bottom: 3px; }}
        .header p {{ font-size: 11px; opacity: 0.9; }}
        .btn-logout {{ position: absolute; right: 10px; top: 10px; background: rgba(255,255,255,0.2); color: white; border: none; padding: 5px 8px; border-radius: 10px; font-size: 10px; cursor: pointer; }}

        /* BUSCADOR */
        .search-section {{ padding: 0 15px; margin-bottom: 15px; }}
        .search-box {{ background: white; border-radius: 12px; padding: 12px; box-shadow: 0 2px 10px rgba(0,0,0,0.08); }}
        .search-input {{ width: 100%; border: none; outline: none; font-size: 15px; font-family: 'Poppins', sans-serif; padding: 8px; }}
        .search-hint {{ font-size: 11px; color: var(--gray); margin-top: 5px; text-align: center; }}
        .db-info {{ background: #e8f5e9; color: #2e7d32; padding: 8px 12px; border-radius: 8px; margin-bottom: 12px; font-size: 11px; text-align: center; }}

        /* RESULTADOS */
        .results-section {{ padding: 0 15px; }}
        .results-header {{ display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px; padding: 0 5px; }}
        .results-count {{ font-size: 12px; color: var(--gray); font-weight: 500; }}
        .pdv-card {{ background: white; border-radius: 12px; padding: 12px; margin-bottom: 10px; box-shadow: 0 2px 8px rgba(0,0,0,0.06); border-left: 4px solid var(--primary); }}
        .card-header {{ display: flex; justify-content: space-between; align-items: flex-start; margin-bottom: 8px; }}
        .pdv-nombre {{ font-size: 14px; font-weight: 700; color: var(--text); flex: 1; padding-right: 8px; }}
        .pdv-codigo {{ background: var(--primary); color: white; padding: 2px 6px; border-radius: 5px; font-size: 10px; font-weight: 600; white-space: nowrap; }}
        .card-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 6px; }}
        .card-field {{ background: #f8f9fa; padding: 6px; border-radius: 5px; }}
        .field-label {{ font-size: 8px; color: var(--gray); text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 2px; }}
        .field-value {{ font-size: 11px; font-weight: 600; color: var(--text); word-break: break-word; }}
        .full-width {{ grid-column: span 2; }}

        .empty-state {{ text-align: center; padding: 40px 20px; color: var(--gray); }}
        .empty-icon {{ font-size: 40px; margin-bottom: 12px; opacity: 0.5; }}

        .session-info {{ position: fixed; bottom: 10px; right: 10px; background: white; padding: 5px 8px; border-radius: 6px; font-size: 9px; color: var(--gray); box-shadow: 0 2px 8px rgba(0,0,0,0.1); }}
    </style>
</head>
<body>
    <!-- LOGIN DISCRETO -->
    <div id="loginOverlay">
        <div class="login-box">
            <div class="login-icon">🔐</div>
            <h2 class="login-title">Acceso</h2>
            <p class="login-subtitle">Base de datos PDV</p>
            <div class="login-error" id="loginError"> Credenciales incorrectas</div>
            <input type="text" class="login-input" id="loginUser" placeholder="Usuario" autocomplete="off">
            <input type="password" class="login-input" id="loginPass" placeholder="Contraseña" maxlength="20" autocomplete="off">
            <button class="login-btn" onclick="validarLogin()">Ingresar</button>
            <div class="login-footer">
                <p>Acceso restringido</p>
            </div>
        </div>
    </div>

    <!-- APP CONTENT -->
    <div id="appContent">
        <div class="header">
            <button class="btn-logout" onclick="cerrarSesion()">Salir</button>
            <h1>📋 PDV Consulta Histórica</h1>
            <p id="userInfo">Usuario</p>
        </div>

        <div class="search-section">
            <div class="db-info" id="dbInfo">
                Base cargada: <strong id="totalPDVs">0</strong> PDVs disponibles
            </div>
            <div class="search-box">
                <input type="text" class="search-input" id="buscador" placeholder="Buscar PDV..." oninput="buscarPDV()">
            </div>
            <div class="search-hint">Busca por Código, Nombre o Número de Recarga</div>
        </div>

        <div class="results-section" id="resultsSection">
            <div class="results-header">
                <div class="results-count" id="resultsCount">0 resultados</div>
            </div>
            <div id="resultsList"></div>
        </div>

        <div class="empty-state" id="emptyState" style="display: none;">
            <div class="empty-icon">🔍</div>
            <p>No se encontraron resultados</p>
        </div>
    </div>

    <div class="session-info" id="sessionInfo" style="display: none;"></div>

    <script>
        const LOGIN_KEY = 'pdvConsultaLogin_v2';
        const USERNAME_KEY = 'pdvConsultaUsuario_v2';
        const DIAS_SESION = 15;
        const EXPIRACION = DIAS_SESION * 24 * 60 * 60 * 1000;

        // DATOS INCRUSTADOS DESDE COLAB
        const todosLosPDVs = {pdv_json};

        let timeoutBusqueda = null;

        // VERIFICAR SESIÓN
        window.onload = function() {{
            const stored = localStorage.getItem(LOGIN_KEY);
            if(stored) {{
                try {{
                    const data = JSON.parse(stored);
                    if(data.expira > Date.now()) {{
                        document.getElementById('loginUser').value = data.usuario || '';
                        iniciarApp();
                    }} else {{
                        localStorage.removeItem(LOGIN_KEY);
                    }}
                }} catch(e) {{
                    localStorage.removeItem(LOGIN_KEY);
                }}
            }}
            document.getElementById('totalPDVs').textContent = todosLosPDVs.length.toLocaleString();
            document.getElementById('buscador').placeholder = `Buscar en ${{todosLosPDVs.length.toLocaleString()}} PDVs...`;
        }};

        function validarLogin() {{
            const user = document.getElementById('loginUser').value.trim();
            const pass = document.getElementById('loginPass').value.trim();
            const error = document.getElementById('loginError');

            // Validación discreta
            if(user.length > 0 && pass.length > 0) {{
                localStorage.setItem(USERNAME_KEY, user);
                localStorage.setItem(LOGIN_KEY, JSON.stringify({{
                    usuario: user,
                    expira: Date.now() + EXPIRACION
                }}));
                iniciarApp();
            }} else {{
                error.style.display = 'block';
                setTimeout(() => {{ error.style.display = 'none'; }}, 3000);
            }}
        }}

        function iniciarApp() {{
            document.getElementById('loginOverlay').style.display = 'none';
            document.getElementById('appContent').style.display = 'block';
            const usuario = document.getElementById('loginUser').value || 'Usuario';
            document.getElementById('userInfo').textContent = usuario;
            mostrarInfoSesion();
        }}

        function mostrarInfoSesion() {{
            const stored = localStorage.getItem(LOGIN_KEY);
            if(stored) {{
                const data = JSON.parse(stored);
                const diasRestantes = Math.ceil((data.expira - Date.now()) / (24 * 60 * 60 * 1000));
                document.getElementById('sessionInfo').style.display = 'block';
                document.getElementById('sessionInfo').textContent = `Sesión: ${{diasRestantes}} días restantes`;
            }}
        }}

        function cerrarSesion() {{
            if(confirm('¿Cerrar sesión?')) {{
                localStorage.removeItem(LOGIN_KEY);
                location.reload();
            }}
        }}

        function buscarPDV() {{
            clearTimeout(timeoutBusqueda);
            const texto = document.getElementById('buscador').value.trim().toLowerCase();

            if(texto.length < 2) {{
                document.getElementById('resultsList').innerHTML = '';
                document.getElementById('resultsCount').textContent = '0 resultados';
                document.getElementById('emptyState').style.display = 'none';
                return;
            }}

            timeoutBusqueda = setTimeout(() => {{
                const resultados = todosLosPDVs.filter(p =>
                    p.codigo.toLowerCase().includes(texto) ||
                    p.nombre.toLowerCase().includes(texto) ||
                    p.recarga.toLowerCase().includes(texto)
                );

                mostrarResultados(resultados);
            }}, 300);
        }}

        function mostrarResultados(resultados) {{
            const container = document.getElementById('resultsList');
            const countEl = document.getElementById('resultsCount');
            const emptyEl = document.getElementById('emptyState');

            countEl.textContent = resultados.length + ' resultado' + (resultados.length !== 1 ? 's' : '');

            if(resultados.length === 0) {{
                container.innerHTML = '';
                emptyEl.style.display = 'block';
                return;
            }}

            emptyEl.style.display = 'none';
            container.innerHTML = resultados.slice(0, 50).map(p => `
                <div class="pdv-card">
                    <div class="card-header">
                        <div class="pdv-nombre">${{escapeHtml(p.nombre)}}</div>
                        <div class="pdv-codigo">${{escapeHtml(p.codigo)}}</div>
                    </div>
                    <div class="card-grid">
                        <div class="card-field">
                            <div class="field-label">Recarga</div>
                            <div class="field-value">${{escapeHtml(p.recarga)}}</div>
                        </div>
                        <div class="card-field">
                            <div class="field-label">Ruta</div>
                            <div class="field-value">${{escapeHtml(p.ruta)}}</div>
                        </div>
                        <div class="card-field">
                            <div class="field-label">Tipo Producto</div>
                            <div class="field-value">${{escapeHtml(p.tipo_producto)}}</div>
                        </div>
                        <div class="card-field">
                            <div class="field-label">Categoría</div>
                            <div class="field-value">${{escapeHtml(p.categoria)}}</div>
                        </div>
                        <div class="card-field">
                            <div class="field-label">Fecha Creación</div>
                            <div class="field-value">${{escapeHtml(p.fecha_creacion)}}</div>
                        </div>
                        <div class="card-field">
                            <div class="field-label">Distrito</div>
                            <div class="field-value">${{escapeHtml(p.distrito)}}</div>
                        </div>
                        <div class="card-field">
                            <div class="field-label">Departamento</div>
                            <div class="field-value">${{escapeHtml(p.departamento)}}</div>
                        </div>
                        <div class="card-field">
                            <div class="field-label">Municipio</div>
                            <div class="field-value">${{escapeHtml(p.municipio)}}</div>
                        </div>
                        <div class="card-field full-width">
                            <div class="field-label">Tipo de Factura</div>
                            <div class="field-value">${{escapeHtml(p.tipo_factura)}}</div>
                        </div>
                        <div class="card-field full-width">
                            <div class="field-label">Tipo de Negocio</div>
                            <div class="field-value">${{escapeHtml(p.tipo_negocio)}}</div>
                        </div>
                    </div>
                </div>
            `).join('');

            if(resultados.length > 50) {{
                container.innerHTML += '<div style="text-align: center; padding: 12px; color: var(--gray); font-size: 11px;">Mostrando los primeros 50 resultados. Refina tu búsqueda.</div>';
            }}
        }}

        function escapeHtml(text) {{
            const div = document.createElement('div');
            div.textContent = text;
            return div.innerHTML;
        }}

        // Permitir Enter para login
        document.addEventListener('keypress', function(e) {{
            if(e.key === 'Enter') {{
                if(document.getElementById('loginOverlay').style.display !== 'none') {{
                    validarLogin();
                }}
            }}
        }});
    </script>
</body>
</html>'''

# Guardar archivo
carpeta = "pdv_consulta_historica"
os.makedirs(carpeta, exist_ok=True)

with open(os.path.join(carpeta, "index.html"), 'w', encoding='utf-8') as f:
    f.write(html_content)

nombre_zip = f"PDV_Consulta_Historica_{datetime.now().strftime('%Y%m%d_%H%M')}.zip"
with zipfile.ZipFile(nombre_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(os.path.join(carpeta, "index.html"), "index.html")

files.download(nombre_zip)

print("\n" + "="*70)
print(" ✅ PDV CONSULTA HISTÓRICA - LISTO")
print("="*70)
print(f"📊 Total PDVs incrustados: {total_pdvs:,}")
print("\n🔐 LOGIN:")
print("   - Usuario: Cualquiera (se guarda)")
print("   - Contraseña: Cualquiera (mínimo 1 caracter)")
print("   - Duración: 15 días")
print("\n📱 CARACTERÍSTICAS:")
print("   ✓ Datos YA están en el HTML (no pide cargar Excel)")
print("   ✓ Login discreto (sin mostrar clave)")
print("   ✓ Búsqueda ultra rápida")
print("   ✓ Todos los campos visibles")
print("="*70)

 PDV CONSULTA HISTÓRICA - PROCESANDO BASE DE DATOS

📤 Selecciona tu archivo Excel con los PDVs:


Saving BASE_PAIS_ABRIL26_SMARTPAY.xlsx to BASE_PAIS_ABRIL26_SMARTPAY (1).xlsx

📊 Total de registros cargados: 19916
✅ 19916 PDVs procesados correctamente

📦 Preparando archivo HTML con datos incrustados...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 ✅ PDV CONSULTA HISTÓRICA - LISTO
📊 Total PDVs incrustados: 19,916

🔐 LOGIN:
   - Usuario: Cualquiera (se guarda)
   - Contraseña: Cualquiera (mínimo 1 caracter)
   - Duración: 15 días

📱 CARACTERÍSTICAS:
   ✓ Datos YA están en el HTML (no pide cargar Excel)
   ✓ Login discreto (sin mostrar clave)
   ✓ Búsqueda ultra rápida
   ✓ Todos los campos visibles
